In previous posts I looked at how to discretize the Stokes equations, which are appropriate for low Reynolds-number fluid flow.
Here I'll look at how we can discretize the full Navier-Stokes equations:
$$\frac{\partial}{\partial t}\rho u + \nabla\cdot\rho u\otimes u = -\nabla p + \nabla\cdot \tau$$
where the deviatoric stress tensor is $\tau = 2\mu\dot\varepsilon$.
I'll use a projection method, which proceeds in two steps: (1) step the velocity forward in time, ignoring the pressure gradient; (2) compute the pressure field by solving the potential equation; and (3) use the newly-computed pressure field to project the candidate velocity from the first step back into the space of divergence-free velocity fields.

### Making the initial geometry

First, we'll make a domain consisting of a hole punched out of a box.

In [ ]:
import gmsh
gmsh.initialize()

In [ ]:
import numpy as np
from numpy import pi as π

Lx = 6.0
Ly = 2.0
lcar = 1 / 16

gmsh.model.add("chamber")
geo = gmsh.model.geo
ps = [(0, 0), (Lx, 0), (Lx, Ly), (0, Ly)]
box_points = [geo.add_point(*p, 0, lcar) for p in ps]
box_lines = [
    geo.add_line(i1, i2) for i1, i2 in zip(box_points, np.roll(box_points, 1))
]

for line in box_lines:
    geo.add_physical_group(1, [line])

f = 1 / 3
c = np.array([f * Lx, Ly / 2, 0])
center = geo.add_point(*c)
r = Ly / 8
num_circle_points = 16
θs = np.linspace(0.0, 2 * π, num_circle_points + 1)[:-1]
ss = np.column_stack((np.cos(θs), np.sin(θs), np.zeros(num_circle_points)))
tie_points = [geo.add_point(*(c + r * s), lcar) for s in ss]
circle_arcs = [
    #geo.add_line(p1, p2)
    geo.add_circle_arc(p1, center, p2)
    for p1, p2 in zip(tie_points, np.roll(tie_points, 1))
]

geo.add_physical_group(1, circle_arcs)

outer_curve_loop = geo.add_curve_loop(box_lines)
inner_curve_loop = geo.add_curve_loop(circle_arcs)
plane_surface = geo.add_plane_surface([outer_curve_loop, inner_curve_loop])
geo.add_physical_group(2, [plane_surface])
geo.synchronize()

In [ ]:
gmsh.model.mesh.generate(2)
gmsh.write("chamber.msh")

In [ ]:
import firedrake
mesh = firedrake.Mesh("chamber.msh")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.set_aspect("equal")
firedrake.triplot(mesh, axes=ax)
ax.legend(loc="upper right");

### Stokes solve

We'll debug things by first solving the Stokes equations.

In [ ]:
cg1 = firedrake.FiniteElement("CG", "triangle", 1)
cg2 = firedrake.FiniteElement("CG", "triangle", 2)
Q = firedrake.FunctionSpace(mesh, cg1)
V = firedrake.VectorFunctionSpace(mesh, cg2)
Z = V * Q
z = firedrake.Function(Z)

In [ ]:
from firedrake import Constant, inner, sym, grad, div, dx, ds

u, p = firedrake.split(z)
μ = Constant(1e-2)
ε = lambda v: sym(grad(v))

p_in, p_out = Constant(1.0), Constant(0.0)
inflow_ids = [1]
outflow_ids = [3]

n = firedrake.FacetNormal(mesh)

L = (
    (μ * inner(ε(u), ε(u)) - p * div(u)) * dx +
    p_in * inner(u, n) * ds(tuple(inflow_ids)) +
    p_out * inner(u, n) * ds(tuple(outflow_ids))
)

In [ ]:
side_wall_ids = [2, 4, 5]
bcs = firedrake.DirichletBC(Z.sub(0), Constant((0.0, 0.0)), side_wall_ids)

In [ ]:
F = firedrake.derivative(L, z)
firedrake.solve(F == 0, z, bcs)

In [ ]:
u, p = z.subfunctions

fig, ax = plt.subplots()
ax.set_aspect("equal")
colors = firedrake.tripcolor(p, axes=ax)
fig.colorbar(colors, orientation="horizontal");

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
colors = firedrake.streamplot(u, resolution=1/16, seed=1729, axes=ax)
fig.colorbar(colors, orientation="horizontal");

### Projection methods

The idea of projection methods is to break up the hard problem of solving the Navier-Stokes equations into two easier problems.
First, we step the velocity field forward in time but assuming the pressure field is fixed.

$$\rho\frac{u_* - u_n}{\delta t} + \nabla\cdot \rho u_n\otimes u_* = -\nabla p_n + \nabla\cdot\tau_*$$

This step produces an approximation to the velocity which might fail to be divergence-free.
We then correct this intermediate approximation by solving a potential equation for the new pressure.

In [ ]:
from firedrake import outer
import irksome
from irksome import Dt

v, q = firedrake.TestFunctions(Z)
u, p = firedrake.split(z)
z_0 = z.copy(deepcopy=True)

F_1 = (
    inner(Dt(u), v) -
    inner(outer(u, u), ε(v)) -
    p * div(v) -
    q * div(u) +
    2 * μ * inner(ε(u), ε(v))
) * dx

F_2 = (
    p_in * inner(v, n) * ds(tuple(inflow_ids)) +
    p_out * inner(v, n) * ds(tuple(outflow_ids))
)

F = F_1 + F_2

In [ ]:
dg0 = firedrake.FiniteElement("DG", "triangle", 0)
Δ = firedrake.FunctionSpace(mesh, dg0)
area = firedrake.Function(Δ).project(firedrake.CellVolume(mesh))
δx_min = np.sqrt(2 * area.dat.data_ro.min())

u, p = z.subfunctions
U_2 = firedrake.Function(Δ).project(inner(u, u))
u_max = np.sqrt(U_2.dat.data_ro.max())
print(δx_min, u_max)

In [ ]:
method = irksome.BackwardEuler()
t = firedrake.Constant(0.0)
dt = firedrake.Constant(0.5 * δx_min / u_max)

params = {
    "solver_parameters": {
        "snes_atol": 1e-12,
        "snes_type": "newtonls",
        "snes_linesearch_type": "l2",
    },
    "bcs": bcs,
}
solver = irksome.TimeStepper(F, method, t, dt, z, **params)

In [ ]:
from tqdm.notebook import trange

zs = [z.copy(deepcopy=True)]

final_time = 10.0
num_steps = int(final_time / float(dt))
progress_bar = trange(num_steps)
for step in progress_bar:
    solver.advance()
    zs.append(z.copy(deepcopy=True))
    iter_count = solver.solver.snes.getIterationNumber()
    progress_bar.set_description(f"iter count: {iter_count}")

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
u, p = z.subfunctions
colors = firedrake.tripcolor(zs[-1].subfunctions[1], axes=ax)
fig.colorbar(colors, orientation="horizontal");

In [ ]:
δp = firedrake.Function(Q).interpolate(zs[-1].subfunctions[1] - zs[0].subfunctions[1])

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
u, p = zs[-1].subfunctions
colors = firedrake.tripcolor(δp, cmap="managua", axes=ax)
fig.colorbar(colors, orientation="horizontal");

In [ ]:
point = np.array([Lx / 2, Ly / 2])
ps = np.array([z.subfunctions[1].at(point) for z in zs])

In [ ]:
ts = np.linspace(0.0, final_time, num_steps + 1)
fig, ax = plt.subplots()
ax.set_xlabel("time")
ax.set_ylabel("pressure")
ax.plot(ts, ps);

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
colors = firedrake.streamplot(u, resolution=1/16, seed=1729, axes=ax)
fig.colorbar(colors, orientation="horizontal");

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect("equal")
arrows = firedrake.quiver(zs[-1].subfunctions[0], axes=ax)
fig.colorbar(arrows, orientation="horizontal");